# Silver Exploration Notebook


## Imports

In [43]:
import os
import findspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    countDistinct,
    percentile_approx,
    radians,
    sin,
    cos,
    asin,
    sqrt,
    pow,
    when,
    floor,
    sum as spark_sum
)

## Start Spark

In [2]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 10:47:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read Bike-sharing dataset
+ Adding city and month

In [4]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

+----+----------+
|city|year_month|
+----+----------+
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
+----+----------+
only showing top 5 rows



## 1. Exploration of Bike-sharing dataset

In [5]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)
 |-- src: string (nullable = false)
 |-- city: string (nullable = true)
 |-- year_month: string (nullable = true)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------

Total rows: 8487065
Columns: ['started_at', 'ended_at', 'duration', 'start_station_id', 'start_station_name', 'start_station_description', 'start_station_latitude', 'start_station_longitude', 'end_station_id', 'end_station_name', 'end_station_description', 'end_station_latitude', 'end_station_longitude', 'src', 'city', 'year_month']


- We have more than 8 470 000 bike trips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `ended_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We have also added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as strings. This is because PySpark reads CSV columns as strings by default. However, several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> double
- Coordinates: `latitude`and `longitude`-> double

#### What about the stations IDs?
- Station IDs: `start_station_id` and `stop_station_id` -> keep as strings or convert to integers?

From the table above we can see the ID is stored as a numeric value. However, we would argue to keep them as a string, as they are identifiers rather than numeric measurements. This also provides greater flexibility if future datasets contain letters(e.g. ST123), without requiring changes to the data type. 


In [6]:
trips_silver = trips_raw

## 2. Converting data types

### duration: string -> double

In [7]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()

0

### started_at / ended_at: string -> timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [8]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

+--------------------------------+-----------------------+--------------------------------+-----------------------+
|started_at                      |started_ts             |ended_at                        |ended_ts               |
+--------------------------------+-----------------------+--------------------------------+-----------------------+
|2022-08-01 03:08:45.036000+00:00|2022-08-01 03:08:45.036|2022-08-01 03:11:38.571000+00:00|2022-08-01 03:11:38.571|
|2022-08-01 03:09:05.419000+00:00|2022-08-01 03:09:05.419|2022-08-01 03:18:15.885000+00:00|2022-08-01 03:18:15.885|
|2022-08-01 03:22:25.178000+00:00|2022-08-01 03:22:25.178|2022-08-01 03:30:57.051000+00:00|2022-08-01 03:30:57.051|
|2022-08-01 03:29:20.355000+00:00|2022-08-01 03:29:20.355|2022-08-01 03:38:43.156000+00:00|2022-08-01 03:38:43.156|
|2022-08-01 03:29:25.883000+00:00|2022-08-01 03:29:25.883|2022-08-01 03:37:00.311000+00:00|2022-08-01 03:37:00.311|
|2022-08-01 03:30:15.563000+00:00|2022-08-01 03:30:15.563|2022-08-01 03:

0

### latitude/ longitude: string -> double

In [9]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()


0

## 3. Missing values and Duplicates

In [10]:
trips_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_silver.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+----------+--------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|src|city|year_month|started_ts|ended_ts|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+----------+--------+
|         0|       0|       0|               0|                 0|                    11240|                     0|                      0|             0|         

#### Missing values
The `start_station_description` column contains around 11 000 missing values, while `end_station_description` contains around 7000 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysis we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station IDs and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to our final analytical dataset. 

### Investigating duplicates

In [11]:
print("Total rows:", trips_silver.count())
print("Duplicate rows:", trips_silver.count() - trips_silver.distinct().count())

Total rows: 8487065


26/11/26 10:48:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:48:42 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


From more than 8 472 000 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We cannot be certain that these are duplicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 columns, including timestamp with microsecond precision. It is highly unlikely that separate trips would share all these values.

We therefore consider these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contains over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset were significantly smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [12]:
rows_in = trips_raw.count()

duplicate_cols = [
    c for c in trips_silver.columns
    if c not in ["src", "year_month"]
]

trips_silver = trips_silver.dropDuplicates(duplicate_cols)

rows_out = trips_silver.count()

duplicated_rows = rows_in - rows_out

print(f"Total rows before: {rows_in}")
print(f"Total rows after: {rows_out}")
print(f"Duplicates removed: {rows_in} - {rows_out} = {duplicated_rows}")

26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:49:31 WARN RowBasedKeyValueBatch: Calling spill() on

Total rows before: 8487065
Total rows after: 8487040
Duplicates removed: 8487065 - 8487040 = 25


### Why not flag the duplicates?
Unlike potentially unrealistic data, duplicate records may lead to double-counting when aggregating bike usage. We record removed duplicates as a quality metric to maintain transparency. 

## 4. Timestamp Validation

### Investigating the timestamp format

In [13]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_ts", "ended_ts", "duration").distinct().limit(10).show(truncate=False)

26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:16 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/26 10:50:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:50:28 WARN RowBasedKeyValueBatch: Calling spill() on

+---+-------+
|len|  count|
+---+-------+
| 25|   8244|
| 32|8478796|
+---+-------+



+-----------------------+-----------------------+--------+
|started_ts             |ended_ts               |duration|
+-----------------------+-----------------------+--------+
|2022-08-01 05:08:33.482|2022-08-01 05:15:55.367|441.0   |
|2022-08-01 06:14:55.037|2022-08-01 06:20:11.933|316.0   |
|2022-08-01 06:44:50.417|2022-08-01 06:49:10.953|260.0   |
|2022-08-01 09:40:18.173|2022-08-01 09:47:06.29 |408.0   |
|2022-08-01 12:13:52.648|2022-08-01 12:17:03.573|190.0   |
|2022-08-01 12:18:20.356|2022-08-01 12:24:31.094|370.0   |
|2022-08-01 13:04:34.948|2022-08-01 13:11:00.678|385.0   |
|2022-08-01 13:18:18.893|2022-08-01 13:45:36.013|1637.0  |
|2022-08-01 15:12:49.362|2022-08-01 15:15:39.084|169.0   |
|2022-08-01 15:20:54.564|2022-08-01 15:26:07.25 |312.0   |
+-----------------------+-----------------------+--------+



### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [14]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp("started_ts", "Europe/Oslo")
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp("ended_ts", "Europe/Oslo")
    )
    
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:13 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/26 10:51:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:27 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_ts             |started_local          |ended_ts               |ended_local            |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 05:08:54.318|2022-08-01 07:08:54.318|2022-08-01 05:29:46.618|2022-08-01 07:29:46.618|
|2022-08-01 05:38:07.074|2022-08-01 07:38:07.074|2022-08-01 05:43:32.596|2022-08-01 07:43:32.596|
|2022-08-01 06:16:17.08 |2022-08-01 08:16:17.08 |2022-08-01 06:21:09.574|2022-08-01 08:21:09.574|
|2022-08-01 06:30:52.895|2022-08-01 08:30:52.895|2022-08-01 06:34:08.804|2022-08-01 08:34:08.804|
|2022-08-01 06:36:13.146|2022-08-01 08:36:13.146|2022-08-01 06:50:21.527|2022-08-01 08:50:21.527|
|2022-08-01 06:44:22.978|2022-08-01 08:44:22.978|2022-08-01 06:50:10.27 |2022-08-01 08:50:10.27 |
|2022-08-01 07:58:42.252|2022-08-01 09:58:42.252|2022-08-01 08:03:12.955|2022-08-01 10:03:12.955|
|2022-08-01 08:14:23

#### Does duration correspond to the start and end time?

In [15]:
trips_silver.select(
    "started_ts",
    "ended_ts",
    "duration",
    # does started_ts and ended_ts match the duration?
    (
        (
            unix_micros("ended_ts") -
            unix_micros("started_ts")
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)

26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:51:47 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+--------+--------------+
|started_ts             |ended_ts               |duration|duration_check|
+-----------------------+-----------------------+--------+--------------+
|2022-08-01 05:08:54.318|2022-08-01 05:29:46.618|1252.0  |1252.3        |
|2022-08-01 05:38:07.074|2022-08-01 05:43:32.596|325.0   |325.522       |
|2022-08-01 06:16:17.08 |2022-08-01 06:21:09.574|292.0   |292.494       |
|2022-08-01 06:30:52.895|2022-08-01 06:34:08.804|195.0   |195.909       |
|2022-08-01 06:36:13.146|2022-08-01 06:50:21.527|848.0   |848.381       |
+-----------------------+-----------------------+--------+--------------+
only showing top 5 rows



From the table above it looks like the duration is always rounded down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [16]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros("ended_ts")
         - unix_micros("started_ts")) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:15 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/26 10:52:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:52:27 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+-------------------+
|summary|      duration_diff|
+-------+-------------------+
|  count|            8487040|
|   mean|0.45516674953811864|
| stddev| 0.2851532667365034|
|    min|                0.0|
|    25%|0.21199999999998909|
|    50%| 0.4240000000000066|
|    75%| 0.6959999999999127|
|    max| 0.9990000000000236|
+-------+-------------------+



Since the maximum difference is below one second, we consider the recorded durations consistent with the timestamps. 

## 5. Coordinate Validation

#### Valid coordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [17]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

0

In [18]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

+-----------------+------------------+-----------------+-----------------+
|          min_lat|           max_lat|          min_lon|          max_lon|
+-----------------+------------------+-----------------+-----------------+
|59.89843428245928|63.446059058709096|5.273174034250616|10.81431433498766|
+-----------------+------------------+-----------------+-----------------+



The latitude and longitude values appear to be reasonable based on the geographical locations of the three cities. Oslo is the southernmost city, while Trondheim is the northernmost city. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

## 6. Trip Duration Analysis

In [19]:
trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


26/11/26 10:53:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:25 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+--------+
|summary|duration|
+-------+--------+
|    min|     0.0|
|    25%|   340.0|
|    50%|   523.0|
|    75%|   805.0|
|    max| 79790.0|
+-------+--------+



26/11/26 10:53:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:53:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:01 WARN RowBasedKeyValueBatch: Calling spill() on

+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1335|            0|
+------------+---------+-------------+



#### Durations to investigate further
- 2 trips have a negative or a duration of 0.
- Is it possible that a trip lasts around 22 hours? (79790/3600 = 22)

In [20]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

+--------------------------------+--------------------------------+--------+----------------+------------------------+-----------------------------------------------+----------------------+-----------------------+--------------+-----------------------+------------------------------------------------------------------+--------------------+---------------------+------------------------------------------------------------------------------+---------+----------+-----------------------+-----------------------+-----------------------+-----------------------+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name      |start_station_description                      |start_station_latitude|start_station_longitude|end_station_id|end_station_name       |end_station_description                                           |end_station_latitude|end_station_longitude|src                                                                           |

How many trips have a duration of 1 second or less? 29


### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need further investigation.

Possible solution: 
- Find the mean/median trip duration from one station to the other and compare it to minimum duration record.
- Flag trips under a specific timeframe. 
- Anything else?

In [21]:
station_durations = (
    trips_silver
    .groupBy("city", "start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration"),
        percentile_approx("duration", 0.5).alias("median_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:54:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:02 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/26 10:55:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:29 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------------+--------------+---------+------------------+------------+---------------+
|city     |start_station_id|end_station_id|num_trips|mean_duration     |min_duration|median_duration|
+---------+----------------+--------------+---------+------------------+------------+---------------+
|bergen   |217             |815           |373      |241.01072386058982|0.0         |150.0          |
|oslo     |465             |523           |94       |581.8723404255319 |0.0         |356.0          |
|bergen   |132             |157           |1220     |542.7598360655737 |1.0         |491.0          |
|bergen   |157             |819           |3083     |222.01751540707104|1.0         |162.0          |
|bergen   |212             |83            |218      |457.76605504587155|1.0         |372.0          |
|bergen   |3               |817           |786      |439.76972010178116|1.0         |341.0          |
|bergen   |788             |808           |1001     |409.4725274725275 |1.0       

### What does the table above tell us?
Although the mean and median durations do not actually tell us the minimum possible duration between two stations, they provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundreds and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only zero or one second. This strongly suggests that some values are incorrect or unrealistic. 

We also see that the mean duration is constantly higher than the median for the station pairs, suggesting that long trips may influence the mean. The median may therefore provide a more robust reference for identifying unusually short trips.

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's at least flag trips with a duration smaller than one second. Further investigation is needed to determine how many trips have a duration significantly below the median of their station pairs and whether these records could affect downstream analyses. 

In [22]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

In [23]:
total_trips = trips_silver.count()

trips_silver = trips_silver.join(
    station_durations.select(
        "city",
        "start_station_id",
        "end_station_id",
        "median_duration",
        "num_trips"
    ),
    on = ["city", "start_station_id", "end_station_id"],
    how = "left"
)

for threshold in [0.01, 0.05, 0.10, 0.20]:
    short_trips = trips_silver.filter(
        (col("num_trips") >= 10) &
        (col("median_duration") > 0) &
        (col("duration") < col("median_duration") * threshold)
    ).count()

    percentage = (short_trips / total_trips) * 100

    print(f"Below {threshold:.0%} of median: " 
          f"{short_trips} trips ({percentage:.6f})")

26/11/26 10:55:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:55:51 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:56:21 WARN RowBasedKeyValueBatch: Calling spill() on

Below 1% of median: 56 trips (0.000660)


26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:58:12 WARN RowBasedKeyValueBatch: Calling spill() on

Below 5% of median: 1235 trips (0.014552)


26/11/26 10:59:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 10:59:47 WARN RowBasedKeyValueBatch: Calling spill() on

Below 10% of median: 11022 trips (0.129869)


26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:01:20 WARN RowBasedKeyValueBatch: Calling spill() on

Below 20% of median: 36934 trips (0.435181)


The results: 
- Below 1% of median: 56 trips (0.000661)
- Below 5% of median: 1230 trips (0.014519)
- Below 10% of median: 10995 trips (0.129785)
- Below 20% of median: 36769 trips (0.434021)

This investigated only station pairs with a minimum of ten recorded trips and a positive median duration.

Since a short trip duration relative to the median does not necessarily indicate an invalid trip, introducing a higher threshold could incorrectly flag valid trips. 

To investigate this further, we can find the straight-line distance between the stations pairs and use the recorded duration to estimate the minimum required average speed. 

### Calculate straight-line distance

In [24]:
# Converting coordinates from degrees to radians
lat1 = radians(col("start_station_latitude"))
lat2 = radians(col("end_station_latitude"))

lon1 = radians(col("start_station_longitude"))
lon2 = radians(col("end_station_longitude"))

# Haversine formula
a = (
    pow(sin((lat2 - lat1) / 2), 2) +
    cos(lat1) * cos(lat2) *
    pow(sin((lon2 - lon1) / 2), 2)
)

# Calulate straight-lin distance in kilometers
trips_silver = trips_silver.withColumn(
    "distance_km",
     2 * 6371 * asin(sqrt(a))
)


trips_silver.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km"
).show(20, truncate=False)

26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:02:56 WARN RowBasedKeyValueBatch: Calling spill() on

+----+----------------+--------------+--------+---------------+------------------+
|city|start_station_id|end_station_id|duration|median_duration|distance_km       |
+----+----------------+--------------+--------+---------------+------------------+
|oslo|1023            |737           |484.0   |448.0          |1.66168957419081  |
|oslo|1755            |423           |811.0   |821.0          |1.994212316899315 |
|oslo|1755            |616           |318.0   |593.0          |1.0345455940791846|
|oslo|2270            |472           |1250.0  |806.0          |2.745137890957619 |
|oslo|2304            |594           |256.0   |212.0          |0.3903005008821124|
|oslo|2305            |2305          |4772.0  |103.0          |0.0               |
|oslo|2306            |462           |1534.0  |1270.0         |3.9756768003320735|
|oslo|2307            |466           |896.0   |323.0          |0.9794777829857129|
|oslo|2315            |481           |514.0   |512.0          |1.4152736456548132|
|osl

### Calculate the Speed

In [25]:
trips_silver = trips_silver.withColumn(
    "speed_kmh",
    when(
        (col("duration") > 0) &
        (col("distance_km").isNotNull()),
        (col("distance_km") * 3600) / col("duration")
    )
)

trips_silver.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km",
    "speed_kmh"
).orderBy(
    col("speed_kmh").desc()
).show(20, truncate=False)

26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:04:21 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------------+--------------+--------+---------------+------------------+------------------+
|city     |start_station_id|end_station_id|duration|median_duration|distance_km       |speed_kmh         |
+---------+----------------+--------------+--------+---------------+------------------+------------------+
|trondheim|268             |2301          |1.0     |1203.0         |3.336113549695287 |12010.008778903033|
|bergen   |819             |221           |1.0     |1098.0         |2.8276269840827126|10179.457142697765|
|bergen   |98              |789           |1.0     |804.0          |2.662220805823626 |9583.994900965054 |
|bergen   |2336            |807           |1.0     |988.0          |2.6011879979085726|9364.27679247086  |
|bergen   |7               |645           |1.0     |1004.0         |2.4739036488273   |8906.05313577828  |
|trondheim|155             |1882          |1.0     |647.0          |2.3508226545398085|8462.96155634331  |
|trondheim|1878            |18       

#### Unrealistic Cycling Speeds
According to vegvesenet, electric bicycles must stop providing motor assistance when reaching a speed of 25 km/h. You may exceed this speed through pedalling or downhill cycling. A speed over 100 km/h is highly unlikely. A speed of around 12 000 km/h must be some type of record, as it's more than twice of the speed of most planes. 

Since our calculated speed is based on the straight-line distance between stations, it therefore represents a lower bound for the actual cycling speed. To investigate potential unrealistic trips, we can evaluate different speed thresholds.

In [26]:
for threshold in [25, 30, 40, 50, 60, 100]:
    fast_trips = trips_silver.filter(
        col("speed_kmh") > threshold
    ).count()

    percentage_speed = (fast_trips / total_trips) * 100

    print(f"Above {threshold} km/h "
          f"{fast_trips} trips ({percentage_speed:.6f}%)")

Above 25 km/h 1788 trips (0.021067%)


Above 30 km/h 1437 trips (0.016932%)


Above 40 km/h 1168 trips (0.013762%)


Above 50 km/h 1004 trips (0.011830%)


Above 60 km/h 875 trips (0.010310%)


Above 100 km/h 568 trips (0.006693%)


The results show that 1786 trips have a mimimum average speed exceeding 25 km/h. More notably 874 trips exceed 60 km/h and 567 trips exceed 100 km/h.

Even though these records represent a small proportion of the dataset, the extremely high estimated speeds strongly suggest data quality issues. 

Compared to the median-based approach, the stright-line distance method provides greater certanty that the trips are in fact unrealistic. 

We could therefore introduce an additional speed-based quality flag to improve the reliability. 

In [27]:
additional_invalid_trips = trips_silver.filter(
    (col("speed_kmh") > 60) &
    (col("duration") <= 1)
).count()

print(f"Trips that are both a speed over 60 km/h and 1 secound: " f"{additional_invalid_trips}")

Trips that are both a speed over 60 km/h and 1 secound: 27


We know that there are 29 trips with a duration with one second or less. We also know that there are 874 trips with a speed over 60 km/h and now we know that 27 of those trips around 1 second are included in the trips that are over a speed of 60 km/h. 

We would therefore conclude with that we can have both instances, both flag trips around one second and flag trips with a speed over 60 km/h. 

In [28]:
# Flag trips that have a speed over 50 km/h
trips_silver = trips_silver.withColumn(
    "unrealistic_speed_flag",
    col("speed_kmh") > 60
)

### Trips with a long duration

In [29]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


26/11/26 11:08:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:08:10 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2026-06-19 13:38:13.233000+00:00|2026-06-20 11:48:03.076000+00:00|79790.0 |
|2025-10-17 14:35:20.232000+00:00|2025-10-18 11:15:52.091000+00:00|74431.0 |
|2022-04-16 12:47:44.363000+00:00|2022-04-17 07:37:10.178000+00:00|67765.0 |
|2023-04-07 13:40:11.612000+00:00|2023-04-08 08:18:37.147000+00:00|67105.0 |
|2023-05-31 15:31:03.704000+00:00|2023-06-01 08:37:01.096000+00:00|61557.0 |
|2026-05-04 18:28:16.356000+00:00|2026-05-05 11:14:23.840000+00:00|60367.0 |
|2022-11-01 15:19:05.897000+00:00|2022-11-02 07:26:44.435000+00:00|58058.0 |
|2022-11-24 14:20:12.725000+00:00|2022-11-25 05:44:40.784000+00:00|55468.0 |
|2026-08-06 20:38:28.625000+00:00|2026-08-07 11:37:01.191000+00:00|53913.0 |
|2022-05-20 16:49:31.692000+00:00|2022-05-21 06:44:45.835000+00:00|50114.0 |

Could a city bike realistically be used for appoximately 22 hours?

According to Oslo Bysykkel, a fully charged electric bike has an estimated range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explanations for the long duration: 
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [30]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

Over 3 hours: 3186


Over 6 hours: 315


Over 12 hours: 15


### Percentage of trips over 3 hours 

In [31]:
total_trips = trips_silver.count()

long_trips = trips_silver.filter(
    col("duration") > 3 * 3600 
).count()

print(f"{long_trips / total_trips * 100}%")

26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:09:21 WARN RowBasedKeyValueBatch: Calling spill() on

0.03753958977452681%


(3184/8470330) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, indicating such trips are extremely rare.

## 7. Trip Distribution

### Number of trips divided by the different cities

In [32]:
trips_silver.groupBy("city").count().show()

26/11/26 11:10:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:11 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+-------+
|     city|  count|
+---------+-------+
|trondheim| 815773|
|     oslo|5766680|
|   bergen|1904587|
+---------+-------+



Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

#### Number of trips divided by city and stations

In [33]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:10:53 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------------+--------------+-----+
|     city|start_station_id|end_station_id|count|
+---------+----------------+--------------+-----+
|     oslo|             586|           607|  486|
|     oslo|             582|           617|  381|
|     oslo|             533|           435|   60|
|     oslo|             443|           444|  572|
|     oslo|             617|          2315|   96|
|     oslo|             462|           545|   51|
|     oslo|             449|           970|   17|
|     oslo|             460|          2333|  426|
|     oslo|             613|           429|  385|
|     oslo|             381|           465|  311|
|     oslo|             608|           540|  156|
|     oslo|             406|           583|  320|
|     oslo|            1755|           519|   75|
|     oslo|             387|           442|  274|
|     oslo|             567|           551|  132|
|     oslo|             474|           493|   56|
|     oslo|            2306|           411|  145|


When we observe the table above we see that:
- Some trip-pair have a significantly higher number of trips than others.
- Some trips start and end at the same station. 

Something else that caught my eye is that the different cities share the same or similar station ID distributions, this may indicate that the different cities use the same identifications. So maybe the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit further:

In [34]:
# Combine start and end station IDs
stations = trips_silver.select(
    "city",
    col("start_station_id").alias("station_id")
).union(
    trips_silver.select(
        "city",
        col("end_station_id").alias("station_id")
    )
).distinct()

# Find station IDs appearing in multiple cities
stations.groupBy("station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:11:35 WARN RowBasedKeyValueBatch: Calling spill() on

+----------+----------+
|station_id|num_cities|
+----------+----------+
+----------+----------+



Surpricingly enough, the investigation shows that no `start_station_id` or `end_station_id` is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential overlaps in future data and it gives a better overview. 

It did however irritate me that I can't see a direct pattern between the cities and their stations, so let's see if there is one:

In [35]:
stations = (
    stations
    .select(
        "city",
        col("station_id").cast("int").alias("station_id")
    )
    .distinct()
)

for n in [2, 3, 4, 5, 10]:
    print(f"\nModulo {n}")

    stations.groupBy(
        "city",
        (col("station_id") % n).alias("remainder")
    ).agg(
        count("*").alias("num_stations")
    ).orderBy("city", "remainder").show()


Modulo 2


26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:12:25 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+---------+------------+
|     city|remainder|num_stations|
+---------+---------+------------+
|   bergen|        0|          72|
|   bergen|        1|          58|
|     oslo|        0|         141|
|     oslo|        1|         150|
|trondheim|        0|          48|
|trondheim|        1|          42|
+---------+---------+------------+


Modulo 3


26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:13:16 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+---------+------------+
|     city|remainder|num_stations|
+---------+---------+------------+
|   bergen|        0|          44|
|   bergen|        1|          46|
|   bergen|        2|          40|
|     oslo|        0|         102|
|     oslo|        1|          93|
|     oslo|        2|          96|
|trondheim|        0|          26|
|trondheim|        1|          36|
|trondheim|        2|          28|
+---------+---------+------------+


Modulo 4


26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:04 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+---------+------------+
|     city|remainder|num_stations|
+---------+---------+------------+
|   bergen|        0|          36|
|   bergen|        1|          26|
|   bergen|        2|          36|
|   bergen|        3|          32|
|     oslo|        0|          69|
|     oslo|        1|          77|
|     oslo|        2|          72|
|     oslo|        3|          73|
|trondheim|        0|          24|
|trondheim|        1|          26|
|trondheim|        2|          24|
|trondheim|        3|          16|
+---------+---------+------------+


Modulo 5


26/11/26 11:14:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:14:53 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+---------+------------+
|     city|remainder|num_stations|
+---------+---------+------------+
|   bergen|        0|          22|
|   bergen|        1|          27|
|   bergen|        2|          30|
|   bergen|        3|          28|
|   bergen|        4|          23|
|     oslo|        0|          61|
|     oslo|        1|          56|
|     oslo|        2|          59|
|     oslo|        3|          59|
|     oslo|        4|          56|
|trondheim|        0|          20|
|trondheim|        1|          24|
|trondheim|        2|          11|
|trondheim|        3|          17|
|trondheim|        4|          18|
+---------+---------+------------+


Modulo 10


26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:15:42 WARN RowBasedKeyValueBatch: Calling spill() on

+------+---------+------------+
|  city|remainder|num_stations|
+------+---------+------------+
|bergen|        0|          12|
|bergen|        1|          11|
|bergen|        2|          15|
|bergen|        3|          11|
|bergen|        4|          12|
|bergen|        5|          10|
|bergen|        6|          16|
|bergen|        7|          15|
|bergen|        8|          17|
|bergen|        9|          11|
|  oslo|        0|          31|
|  oslo|        1|          29|
|  oslo|        2|          28|
|  oslo|        3|          30|
|  oslo|        4|          26|
|  oslo|        5|          30|
|  oslo|        6|          27|
|  oslo|        7|          31|
|  oslo|        8|          29|
|  oslo|        9|          30|
+------+---------+------------+
only showing top 20 rows



The station IDs appear to be distributed without an obvious city-specific numerical pattern. Modulo analysis showed relatively balanced distributions across the three cities, with no clear preference for even or odd numbers or particular remainders. However, this does not establish whether IDs were assigned randomly or sequentially. 

Next, we want to investigate whether the stations information associated with each ID remains consistent throughout the dataset. 

In [47]:
start_stations = trips_silver.select(
    "city",
    F.col("start_station_id").alias("station_id"),
    F.col("start_station_name").alias("station_name"),
    F.col("start_station_latitude").alias("latitude"), 
    F.col("start_station_longitude").alias("longitude")
)

end_stations = trips_silver.select(
    "city", 
    F.col("end_station_id").alias("station_id"),
    F.col("end_station_name").alias("station_name"),
    F.col("end_station_latitude").alias("latitude"),
    F.col("end_station_longitude").alias("longitude")
)

stations = (
    start_stations
    .unionByName(end_stations)
    .distinct()
)

In [48]:
station_consistency = (
    stations
    .groupBy("city", "station_id")
    .agg(
        F.countDistinct("station_name").alias("num_names"),
        F.countDistinct("latitude", "longitude").alias("num_locations")
    )
)

station_consistency.filter(
    (F.col("num_names") > 1) | (F.col("num_locations") > 1)
).show(50, truncate=False)

26/11/26 12:04:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:04:04 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------+---------+-------------+
|city     |station_id|num_names|num_locations|
+---------+----------+---------+-------------+
|bergen   |220       |1        |3            |
|oslo     |454       |1        |2            |
|trondheim|14        |2        |4            |
|oslo     |434       |1        |2            |
|oslo     |566       |1        |3            |
|trondheim|85        |1        |3            |
|oslo     |384       |2        |3            |
|trondheim|105       |1        |3            |
|oslo     |612       |1        |2            |
|bergen   |1898      |1        |2            |
|bergen   |640       |1        |2            |
|trondheim|175       |1        |3            |
|trondheim|5426      |1        |2            |
|oslo     |514       |1        |2            |
|bergen   |87        |1        |2            |
|oslo     |599       |1        |2            |
|oslo     |533       |1        |2            |
|bergen   |116       |1        |2            |
|oslo     |44

### Which station pair are the most used?

In [36]:
trips_silver.groupBy(
    "city",
    "start_station_id",
    "end_station_id"
).count().orderBy(
    col("count").desc()
).show(20, truncate=False)

26/11/26 11:16:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:28 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:16:32 WARN RowBasedKeyValueBatch: Calling spill() on

+------+----------------+--------------+-----+
|city  |start_station_id|end_station_id|count|
+------+----------------+--------------+-----+
|bergen|36              |814           |8163 |
|bergen|36              |138           |6550 |
|bergen|814             |36            |6515 |
|bergen|789             |220           |6427 |
|bergen|220             |789           |5192 |
|bergen|138             |36            |4935 |
|oslo  |443             |452           |4625 |
|oslo  |464             |443           |4347 |
|oslo  |456             |479           |4290 |
|oslo  |479             |456           |4240 |
|oslo  |452             |443           |4092 |
|oslo  |478             |452           |3961 |
|bergen|819             |157           |3890 |
|bergen|3396            |24            |3712 |
|bergen|794             |368           |3515 |
|bergen|816             |138           |3473 |
|oslo  |443             |464           |3453 |
|bergen|138             |816           |3433 |
|bergen|150  

In [49]:
stations.groupBy("city").agg(
    F.countDistinct("station_id").alias("num_stations")
).show()

26/11/26 12:10:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 12:10:30 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+------------+
|     city|num_stations|
+---------+------------+
|trondheim|          90|
|     oslo|         291|
|   bergen|         130|
+---------+------------+



#### Number of trips divided by city and month

In [37]:
trips_silver.groupBy("city", "year_month").count().orderBy(
    "city", "year_month"
).show(200)

+---------+----------+------+
|     city|year_month| count|
+---------+----------+------+
|   bergen|   2022-01| 26375|
|   bergen|   2022-02| 28389|
|   bergen|   2022-03| 50234|
|   bergen|   2022-04| 49972|
|   bergen|   2022-05| 56259|
|   bergen|   2022-06| 57101|
|   bergen|   2022-07| 30356|
|   bergen|   2022-08| 60298|
|   bergen|   2022-09| 68782|
|   bergen|   2022-10| 47233|
|   bergen|   2022-11| 42541|
|   bergen|   2022-12| 17773|
|   bergen|   2023-01| 24829|
|   bergen|   2023-02| 30059|
|   bergen|   2023-03| 30451|
|   bergen|   2023-04| 41145|
|   bergen|   2023-05| 49749|
|   bergen|   2023-06| 58606|
|   bergen|   2023-07| 35019|
|   bergen|   2023-08| 52498|
|   bergen|   2023-09| 49709|
|   bergen|   2023-10| 42989|
|   bergen|   2023-11| 36744|
|   bergen|   2023-12| 16062|
|   bergen|   2024-01| 20408|
|   bergen|   2024-02| 19706|
|   bergen|   2024-03| 29804|
|   bergen|   2024-04| 43104|
|   bergen|   2024-05| 56798|
|   bergen|   2024-06| 42879|
|   bergen

Further we want to investigate when people actually use the bikes throughout the day. Are there certain hours when the bike-sharing is more popular, and do these pattern differ between weekdays and weekends?

In [44]:
trips_silver = (
    trips_silver
    .withColumn("local_hour", F.hour("started_local"))
    .withColumn("weekday", F.dayofweek("started_local"))
    .withColumn("is_weekend", F.col("weekday").isin(1,7))
)

trips_silver.groupBy(
    "city", "local_hour"
).count().orderBy(
    "city", "local_hour"
).show(72)

26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:52 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:24:55 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------+------+
|     city|local_hour| count|
+---------+----------+------+
|   bergen|         0|   198|
|   bergen|         1|    31|
|   bergen|         2|    25|
|   bergen|         3|    20|
|   bergen|         4|    16|
|   bergen|         5|  8065|
|   bergen|         6| 38939|
|   bergen|         7|137559|
|   bergen|         8|183073|
|   bergen|         9| 95008|
|   bergen|        10| 78850|
|   bergen|        11| 83921|
|   bergen|        12| 89720|
|   bergen|        13| 95998|
|   bergen|        14|124622|
|   bergen|        15|204502|
|   bergen|        16|202050|
|   bergen|        17|144701|
|   bergen|        18|119044|
|   bergen|        19| 92448|
|   bergen|        20| 75073|
|   bergen|        21| 58217|
|   bergen|        22| 43070|
|   bergen|        23| 29437|
|     oslo|         0| 90473|
|     oslo|         1|  6759|
|     oslo|         2|  7447|
|     oslo|         3|  3327|
|     oslo|         4|  2302|
|     oslo|         5| 25881|
|     oslo

We can observe two distinxt peaks, one around 08:00 and anthoer around 15:00-16:00. This suggesting that a considerable proportion of the trips may be related to commuting. 

The other thing that caught my attention is the difference in bike-sharing usage during the night. Trondheim has no recorded trips between 03:00 and 06:00. The different cities do have different operating hours.  

Opning hours:
- Oslo:           05:00 - 01:00
- Trondheim:      06:00 - 00:00
- Bergen:         05:00 - 00:00

You can only rent a bike in the opening hours, but you can deliver the bike 24/7.

In [46]:
trips_silver.groupBy(
    "city", "is_weekend", "local_hour"
).count().orderBy(
    "city", "is_weekend", "local_hour"
).show(144)

26/11/26 11:27:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:27:13 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------+----------+------+
|     city|is_weekend|local_hour| count|
+---------+----------+----------+------+
|   bergen|     false|         0|   122|
|   bergen|     false|         1|    14|
|   bergen|     false|         2|    11|
|   bergen|     false|         3|     8|
|   bergen|     false|         4|     3|
|   bergen|     false|         5|  7260|
|   bergen|     false|         6| 36719|
|   bergen|     false|         7|133275|
|   bergen|     false|         8|177103|
|   bergen|     false|         9| 83134|
|   bergen|     false|        10| 62266|
|   bergen|     false|        11| 61276|
|   bergen|     false|        12| 63807|
|   bergen|     false|        13| 68248|
|   bergen|     false|        14| 94691|
|   bergen|     false|        15|174640|
|   bergen|     false|        16|173853|
|   bergen|     false|        17|118051|
|   bergen|     false|        18| 95332|
|   bergen|     false|        19| 73078|
|   bergen|     false|        20| 59103|
|   bergen|     

## 8. Summary and proposed Silver tranformations
Based on this exploratory analysis, these are the following transformations we propose for the Silver layer:
- Extract city and year-month information from the source file path.
- Convert trip duration and geographical coordinates to double. 
- Convert start and and timestamps to UTC timestamps.
- Derive Norwegian local timestamps for temporal analysis. 
- Remove duplicate trip records. 
- Preserve station IDs as strings. 
- Preserve missing station descriptions.
- Validate recorded durations against the start and end timestamps.
- Validate geographical coordinates.
- Calculate the minimum required average speed for each trip.
- Flag trips with a recorded duration of one second. 
- Flag trips requiring a minimum average speed exceeding 50 km/h.
- Preserve long-duration trips, as their duration alone does not necessarily indicate a data quality issue. 

## 9. Test silver_trips.py

In [38]:
import sys
import os

print(os.getcwd())
sys.path.append("../Silver")

from silver_trips import (
    read_raw_trips,
    build_silver_trips,
    trips_quality_metrics
)

/home/ubuntu/group11-astrid/Project/notebooks


In [39]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

trips_silver_test = build_silver_trips(spark, RAW)

trips_silver_test.printSchema()
trips_silver_test.show(5, truncate=False)

root
 |-- duration: double (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- started_ts: timestamp (nullable = true)
 |-- ended_ts: timestamp (nullable = true)
 |-- started_local: timestamp (nullable = true)
 |-- ended_local: timestamp (nullable = true)
 |-- short_trip_flag: boolean (nullable = true)
 |-- duration_diff: double (nullable = true)
 |-- invalid_duration_flag: boolean (nullable = true)
 |-- invalid_coordinates_flag:

+--------+----------------+--------------------+-----------------------------------------------+----------------------+-----------------------+--------------+-----------------+-------------------------------------------+--------------------+---------------------+---------+----------+-----------------------+-----------------------+-----------------------+-----------------------+---------------+-------------------+---------------------+------------------------+-----------------------+
|duration|start_station_id|start_station_name  |start_station_description                      |start_station_latitude|start_station_longitude|end_station_id|end_station_name |end_station_description                    |end_station_latitude|end_station_longitude|city     |year_month|started_ts             |ended_ts               |started_local          |ended_local            |short_trip_flag|duration_diff      |invalid_duration_flag|invalid_coordinates_flag|_silver_processed_time |
+--------+--------------

In [40]:
trips_quality_metrics(spark, RAW)

26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/26 11:18:23 WARN RowBasedKeyValueBatch: Calling spill() on

{'layer': 'silver_trips',
 'rows_in': 8487065,
 'rows_out': 8487040,
 'duplicates_removed': 25,
 'short_trips': 29,
 'invalid_duration': 0,
 'invalid_coordinates': 0}